# Lektion 08: Embeddings & Vektor-RAG – Suche nach Bedeutung



| | |
|---|---|
| **Zeit** | ~60 Minuten |
| **Schwierigkeit** | Mittel |
| **Voraussetzungen** | Lektion 05–07 |
| **Du baust** | Einen echten Vektor-Index über deine Dokumente – mit Chroma und Retriever |

## 🎯 Nach dieser Lektion kannst du:

- erklären, warum Wort-Suche an Grenzen stößt („freie Tage" vs. „Urlaub")

- beschreiben, was ein **Embedding** ist (Bedeutungs-Koordinaten)

- Dokumente **chunken** und in Chroma speichern

- mit einem **Retriever** semantisch suchen und Treffer prüfen

- Zwischenergebnisse (Chunks, Embeddings, Treffer) **sichtbar inspizieren**

## ⚙️ Setup – einmal pro Notebook ausführen

Diese Zelle findet die Projektwurzel und lädt die **zentrale KI-Konfiguration** aus deiner `.env` (siehe `docs/local-ai-server.md`). Du musst nie Notebook-Code ändern, um den Anbieter zu wechseln.

In [ ]:
import sys
from pathlib import Path

# Projektwurzel finden (dort liegt der Ordner "lernakademie")
ROOT = Path.cwd()
while not (ROOT / "lernakademie").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "lernakademie").exists():
    raise RuntimeError("Projektwurzel nicht gefunden – starte das Notebook im Repo-Ordner.")
sys.path.insert(0, str(ROOT))

from lernakademie.config import llm_zusammenfassung

_info = llm_zusammenfassung()  # zeigt NIE den API-Key selbst
print("KI-Konfiguration:", _info["anbieter"], "| Modell:", _info["modell"],
      "| Key:", _info["api_key"])
if _info["anbieter"] == "offline":
    print("Offline-Modus: alles laeuft – Antworten sind Simulationen ([OFFLINE]).")


## 🧠 Konzept: Embeddings – die Landkarte der Bedeutung

**WARUM:** Deine Build-1-Suche findet „Urlaub" nur, wenn „Urlaub" wörtlich steht. **Metapher:** Der Nutzer fragt nach "freien Tagen" – dein Wortsucher dreht durch, weil er nur Wort-Gleichheit kennt.

**WAS:** Ein **Embedding** übersetzt Text in Zahlenkoordinaten – einen **Vektor**. Texte mit ähnlicher **Bedeutung** liegen auf dieser Landkarte nah beieinander, **auch ohne ein gemeinsames Wort**. Eine **Vektor-Datenbank** ist die Bibliothek, die nach Bedeutung statt nach Alphabet sortiert.

**Das RAG-Flussbild (merk es dir):**

```text
Dokumente → Chunks → Embeddings → Vektor-Store
Frage → Embedding der Frage
        ↓ (Nachbarschaft auf der Landkarte)
top-k Chunks → Kontext → LLM → Antwort + Quellen
```

In dieser Akademie nutzt `get_embeddings()` je nach Konfiguration echte Mistral-Embeddings oder (offline) ein transparentes Schlüsselwort-Modell – die Pipeline bleibt identisch (siehe `docs/local-ai-server.md`).

## Schritt 1: Chunking

**WARUM:** Ganze Dokumente sind zu groß fürs Kontextfenster und zu unpräzise für die Suche. Wir schneiden sie in **Chunks** (Stückchen).

**Die zwei Parameter – mit Metapher:**
- `chunk_size=600` Zeichen: nicht zu klein (Sinn zerschnitten), nicht zu groß (Suche wird unscharf).
- `chunk_overlap=100`: die Schere überlappt – nichts fällt zwischen die Schnitte.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)

DOKUMENT = (ROOT / "data" / "mitarbeiter-handbuch.md").read_text(encoding="utf-8")
chunks = splitter.split_text(DOKUMENT)

print(f"{len(chunks)} Chunks aus einer Datei")
print("Chunk 1:", chunks[0][:120], "...")
print("Chunk 2:", chunks[1][:120], "...")

### 🔎 Was sehen wir hier?
Der **RecursiveCharacterTextSplitter** schneidet hierarchisch (Absätze → Sätze → Wörter), sodass Chunks natürlich bleiben. **TRY:** Ändere `chunk_size` auf 50 und beobachte, wie der Sinn zerfällt!

## Schritt 2: Embeddings anfassen

Bevor wir die Black Box nutzen, schauen wir **hinein**. (Offline-Modus: die Vektoren sind vereinfachte Schlüsselwort-Vektoren – das Prinzip ist dasselbe.)

In [ ]:
from lernakademie.offline_embeddings import get_embeddings

embeddings = get_embeddings()
print("Embedding-Modell:", type(embeddings).__name__)

fragen = ["Wie viele Urlaubstage habe ich?",
          "Bis wann muss der Urlaubsantrag gestellt werden?",
          "Wie sieht eine sichere Passwort-Regel aus?"]
vektoren = [embeddings.embed_query(f) for f in fragen]

def aehnlichkeit(u, v):
    return sum(a * b for a, b in zip(u, v))   # Punktprodukt ≈ Cosine bei normalisierten Vektoren

print(f"\nAehnlichkeit Frage1/Question2 (beide Urlaub): {aehnlichkeit(vektoren[0], vektoren[1]):.2f}")
print(f"Aehnlichkeit Frage1/Question3 (Urlaub vs Passwort): {aehnlichkeit(vektoren[0], vektoren[2]):.2f}")

### 🔎 Was sehen wir hier?
Themenverwandte Fragen haben **höhere Ähnlichkeit** als fremde – genau darauf baut die Suche. Die Landkarte existiert wirklich, und du hast sie gerade vermessen.

## Schritt 3: Der Vektor-Store (Chroma) & der Retriever

**WAS:** **Chroma** speichert Chunks + Vektoren + **Metadaten** (z. B. die Quelldatei – wichtig für Quellenangaben!). Der **Retriever** ist der einheitliche Such-Schalter darüber.

In [ ]:
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

def baue_vektor_index(k=3):
    """Liest alle data/*.md, chunkt, embedded und liefert einen Retriever."""
    from langchain_community.document_loaders import TextLoader
    splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)
    alle_chunks = []
    for datei in sorted((ROOT / "data").glob("*.md")):
        dokumente = TextLoader(str(datei), encoding="utf-8").load()
        alle_chunks.extend(splitter.split_documents(dokumente))
    db = Chroma(collection_name="lektion08", embedding_function=get_embeddings())
    db.add_documents(alle_chunks)
    return db.as_retriever(search_kwargs={"k": k})

retriever = baue_vektor_index()
print("Index gebaut – frage ihn:")

In [ ]:
treffer = retriever.invoke("Wie viele Urlaubstage gibt es?")
for dokument in treffer:
    print("Quelle:", dokument.metadata.get("source", "?"))
    print("Auszug:", dokument.page_content.strip().replace("\n", " ")[:100], "\n")

## ✏️ Deine Aufgabe: Der Build-1-Killer-Test

Du beweist die Schwäche deiner Build-1-Suche und die Stärke des Retrievers – mit derselben Frage.

**Anforderungen:**

- Stelle die Frage: Wie beantrage ich freie Tage? (a) an `suche_relevant` aus Build 1 und (b) an deinen Retriever.

- Gib beide Trefferlisten aus (Quelle/Titel bzw. Quelle/Auszug).

- Beantworte als Kommentar: Welches Verfahren liefert den relevanten Abschnitt – und warum?

**Schreibe deinen Code in die folgende Zelle:**

In [ ]:
from lernakademie.build1 import lade_dokumente, suche_relevant

# YOUR CODE HERE: (a) Build-1-Suche mit der Frage

# YOUR CODE HERE: (b) Retriever mit derselben Frage

# DEINE BEOBACHTUNG: ...

### ✅ Richtig, wenn:

- [ ] Beide Ergebnisse sind sichtbar gegenübergestellt.

- [ ] Deine Beobachtung erklärt den Unterschied (Wortgleichheit vs. Bedeutung).

**🧪 Fuehre die folgende Zelle aus, um dein Ergebnis zu pruefen:**

In [ ]:
build1_treffer = suche_relevant("Wie beantrage ich freie Tage?", lade_dokumente())
vektor_treffer = retriever.invoke("Wie beantrage ich freie Tage?")
print("Build 1:", [t["titel"] for t in build1_treffer] or "keine Treffer")
print("Retriever:", [d.metadata.get("source", "?") for d in vektor_treffer])
assert vektor_treffer, "Der Retriever sollte Treffer liefern."
print("Vergleich ok ✅ (die inhaltliche Wertung macht dein Kommentar)")

<details><summary>💡 Hinweis 1 (oeffnen, wenn du nicht weiterkommst)</summary>

Für (a): `suche_relevant("Wie beantrage ich freie Tage?", lade_dokumente())`.
</details>

<details><summary>💡 Hinweis 2 (oeffnen, wenn du nicht weiterkommst)</summary>

Für (b): `retriever.invoke("Wie beantrage ich freie Tage?")`.
</details>

<details><summary>💡 Hinweis 3 (oeffnen, wenn du nicht weiterkommst)</summary>

Offline-Modus: Der Schlüsselwort-Retriever erkennt „Urlaub“ über seinen Stamm – schau genau hin, was er findet.
</details>

## 📝 Reflexion

Beantworte in deinen eigenen Worten:

1. Warum darf man Chunks mit verschiedenen Embedding-Modellen **nie** mischen?

2. Was kostet `k=6` gegenüber `k=3` – Latenz, Tokens, Präzision?

## ➡️ Weiter

**🔗 Lektion 09 schließt BUILD 2 ab:** `notebooks/05_build_2/09_build2_rag_assistent.ipynb` – Retriever + Chain + Quellen = dein Assistent in neuer Architektur.